In [8]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

SEED = 111
EMBEDDING_PATH = Path("mobilised_embeddings.npy")


# ---------------------------------------------------------
# 1. Load Participant Embeddings
# ---------------------------------------------------------
def load_participant_embeddings(path):
    day_embeddings = np.load(path, allow_pickle=True).item()
    participants = {}
    for eid, days in day_embeddings.items():
        days = np.asarray(days, dtype=np.float32)
        norms = np.linalg.norm(days, axis=1)
        valid = np.isfinite(norms) & (norms > 0)
        if valid.any():
            participants[str(eid)] = (days[valid] / norms[valid, None]).mean(axis=0)

    frame = pd.DataFrame.from_dict(participants, orient="index")
    frame.index.name = "eid"
    frame.columns = [f"embedding_{column:03d}" for column in range(frame.shape[1])]
    return frame


embeddings_df = load_participant_embeddings(EMBEDDING_PATH)
embedding_cols = embeddings_df.columns.tolist()

# ---------------------------------------------------------
# 2. Load Demographic & Target Data
# ---------------------------------------------------------
demographics_path = os.path.join("..", "data", "demographics.csv")
demographic_df = pd.read_csv(demographics_path)

# Calculate BMI if missing
if "BMI" not in demographic_df.columns and {"weight", "height"}.issubset(demographic_df.columns):
    demographic_df["BMI"] = (demographic_df["weight"] / (demographic_df["height"] ** 2)) * 10000

demographic_cols = ["age", "gender", "BMI", "smoker"]
demographic_df = demographic_df[["Local Participant"] + demographic_cols].copy()
demographic_df["Local Participant"] = demographic_df["Local Participant"].astype(str)

# Load timestamped target data
target_path = os.path.join("..", "data", "timestamped_nep.csv")
target_df = pd.read_csv(target_path)
target_df = target_df[["Local.Participant", "visit.number", "timestamp_plus_one_NEP"]].dropna(
    subset=["timestamp_plus_one_NEP"]
)
target_df["Local.Participant"] = target_df["Local.Participant"].astype(str)

# Merge Demographics with Target Data
df = target_df.merge(
    demographic_df,
    left_on="Local.Participant",
    right_on="Local Participant",
    how="inner",
)

# Clean visit numbers and sort
df["visit.number"] = df["visit.number"].str.upper()
df["visit_num"] = df["visit.number"].str.extract(r"(\d+)").astype(int)
df = df.sort_values(by=["Local.Participant", "visit_num"]).reset_index(drop=True)

# Filter post-progression visits
first_positives = (
    df[df["timestamp_plus_one_NEP"] == 1]
    .groupby("Local.Participant")["visit_num"]
    .min()
    .reset_index()
    .rename(columns={"visit_num": "first_positive_visit"})
)

df = df.merge(first_positives, on="Local.Participant", how="left")
df_filtered = df[
    df["first_positive_visit"].isna() | (df["visit_num"] <= df["first_positive_visit"])
].copy()

# Construct eid matching embedding index format (e.g., "12345_T1")
df_filtered["eid"] = (
    df_filtered["Local.Participant"].astype(str) + "_" + df_filtered["visit.number"]
)

# ---------------------------------------------------------
# 3. Build Full Cohort (Demographics + Embeddings + Target)
# ---------------------------------------------------------
cohort = df_filtered.set_index("eid").join(embeddings_df, how="inner")
print(f"Total samples with complete data: {len(cohort):,}")

# ---------------------------------------------------------
# 4. Define Feature Preprocessors
# ---------------------------------------------------------
num_demo_cols = ["age", "BMI"]
cat_demo_cols = ["gender", "smoker"]

demo_transformer = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_demo_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
        ]), cat_demo_cols)
    ]
)

sensori_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Participant group splitting array
groups = cohort["Local.Participant"]

# ---------------------------------------------------------
# 5. Model Evaluations
# ---------------------------------------------------------
def train_and_evaluate_classifier(X_train, X_test, y_train, y_test, model_name):
    clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)

    print(f"\n=================== {model_name} ===================")
    print("Confusion Matrix:")
    print(f"TN: {cm[0, 0]:<5} FP: {cm[0, 1]:<5}")
    print(f"FN: {cm[1, 0]:<5} TP: {cm[1, 1]:<5}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

y_cls = cohort["timestamp_plus_one_NEP"]
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx_cls, test_idx_cls = next(sgkf.split(cohort, y_cls, groups=groups))

# Preprocess features separately
X_sensori = cohort[embedding_cols]
X_demo = cohort[demographic_cols]

X_sens_train_cls = sensori_transformer.fit_transform(X_sensori.iloc[train_idx_cls])
X_sens_test_cls = sensori_transformer.transform(X_sensori.iloc[test_idx_cls])

X_demo_train_cls = demo_transformer.fit_transform(X_demo.iloc[train_idx_cls])
X_demo_test_cls = demo_transformer.transform(X_demo.iloc[test_idx_cls])

# Combined feature matrices
X_comb_train_cls = np.hstack([X_demo_train_cls, X_sens_train_cls])
X_comb_test_cls = np.hstack([X_demo_test_cls, X_sens_test_cls])

y_train_cls, y_test_cls = y_cls.iloc[train_idx_cls], y_cls.iloc[test_idx_cls]

# Run Classification Models
train_and_evaluate_classifier(
    X_sens_train_cls, X_sens_test_cls, y_train_cls, y_test_cls,
    "Model 1 (Classifier): Sensori Embeddings Only"
)

train_and_evaluate_classifier(
    X_demo_train_cls, X_demo_test_cls, y_train_cls, y_test_cls,
    "Model 2 (Classifier): Demographics Only"
)

train_and_evaluate_classifier(
    X_comb_train_cls, X_comb_test_cls, y_train_cls, y_test_cls,
    "Model 3 (Classifier): Demographics + Sensori Embeddings"
)

Total samples with complete data: 781

=================== Model 1 (Classifier): Sensori Embeddings Only ===================
Confusion Matrix:
TN: 128   FP: 19   
FN: 10    TP: 0    

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.87      0.90       147
         1.0       0.00      0.00      0.00        10

    accuracy                           0.82       157
   macro avg       0.46      0.44      0.45       157
weighted avg       0.87      0.82      0.84       157

ROC-AUC Score: 0.5735

=================== Model 2 (Classifier): Demographics Only ===================
Confusion Matrix:
TN: 80    FP: 67   
FN: 2     TP: 8    

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.54      0.70       147
         1.0       0.11      0.80      0.19        10

    accuracy                           0.56       157
   macro avg       0.54      0.67      0.44       157
weighted 